In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 XGBoost Classifier (`models/xgboost_raw_esi1_extreme.ipynb`)

This notebook trains a **Standard Binary XGBoost Classifier** for **ESI 1 vs Not ESI 1** on natural data distributions using **19 Predictor Features** (`age`, `gender`, `cc_breathingdifficulty` + 16 Continuous Vital Delta & Range Features):

### System Architecture & Workflow
1. **Predictor Feature Selection (19 Predictors)**:
   - **Baseline Raw Features (From `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`.
   - **16 Continuous Vital Delta & Range Features**:
     - `hr_mean_to_last`: `triage_vital_hr - pulse_last`
     - `sbp_mean_to_last`: `triage_vital_sbp - sbp_last`
     - `spo2_mean_to_last`: `triage_vital_o2 - spo2_last`
     - `rr_mean_to_last`: `triage_vital_rr - resp_last`
     - `hr_range`: `pulse_max - pulse_min`
     - `rr_range`: `resp_max - resp_min`
     - `spo2_range`: `spo2_max - spo2_min`
     - `sbp_range`: `sbp_max - sbp_min`
     - `hr_last_to_min`: `pulse_last - pulse_min`
     - `rr_last_to_min`: `resp_last - resp_min`
     - `spo2_last_to_min`: `spo2_last - spo2_min`
     - `sbp_last_to_min`: `sbp_last - sbp_min`
     - `hr_last_to_max`: `pulse_last - pulse_max`
     - `rr_last_to_max`: `resp_last - resp_max`
     - `spo2_last_to_max`: `spo2_last - spo2_max`
     - `sbp_last_to_max`: `sbp_last - sbp_max`
2. **Stratified Data Partitioning**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling on natural data distributions.
3. **Standard Binary XGBoost Gradient Boosting**: Fits binary objective decision trees (`objective = "binary:logistic"`, `eval_metric = "logloss"`) via `xgb.DMatrix` and `xgb.train()`.
4. **Comprehensive Benchmarking Across Splits**: Evaluates Train, Validation, and Test performance with confusion matrices, Class Count Comparison Tables, Accuracy, ESI 1 Precision, ESI 1 Recall (Sensitivity), F1 Score, PR-AUC, and ROC-AUC.
5. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_raw_esi1_metrics_barchart.png`).
   - **CSV Reports**: `reports/xgboost_raw_esi1_val_report.csv`, `reports/xgboost_raw_esi1_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_raw_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 19 Predictor Features (age, gender, cc_bd + 16 Vital Delta/Range Features)
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Helper vectors for vital trends and ranges
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 19 Predictors: age, gender, cc_breathingdifficulty + 16 Continuous Vital Delta & Range Features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  # 16 Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (19 Total):\n")
print(setdiff(names(df_full), "target_layer1"))
cat("\nNatural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning (Standard Natural Distribution)
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous features across splits
cont_cols <- intersect(
  c("age", "hr_mean_to_last", "sbp_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last",
    "hr_range", "rr_range", "spo2_range", "sbp_range",
    "hr_last_to_min", "rr_last_to_min", "spo2_last_to_min", "sbp_last_to_min",
    "hr_last_to_max", "rr_last_to_max", "spo2_last_to_max", "sbp_last_to_max"),
  names(train_df)
)
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
cat("=== Natural Target Distributions Across Splits ===\n")
cat("Natural Training Set Target Distribution:\n")
print(table(train_df$target_layer1))
cat("\nNatural Validation Set Target Distribution:\n")
print(table(val_df$target_layer1))
cat("\nNatural Test Set Target Distribution:\n")
print(table(test_df$target_layer1))
feat_names <- setdiff(names(train_df), "target_layer1")
X_train <- as.matrix(train_df[, feat_names])
y_train <- ifelse(train_df$target_layer1 == "1", 1, 0)
X_val   <- as.matrix(val_df[, feat_names])
y_val   <- ifelse(val_df$target_layer1 == "1", 1, 0)
X_test  <- as.matrix(test_df[, feat_names])
y_test  <- ifelse(test_df$target_layer1 == "1", 1, 0)
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train)
dval_xgb   <- xgb.DMatrix(data = X_val, label = y_val)
dtest_xgb  <- xgb.DMatrix(data = X_test, label = y_test)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary XGBoost Model on Natural Distribution
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training Binary XGBoost Model on Natural Data (ESI 1 vs Not ESI 1)...\n")
xgb_params <- list(
  objective        = "binary:logistic",
  eval_metric      = "logloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params    = xgb_params,
  data      = dtrain_xgb,
  nrounds   = 150,
  evals     = list(train = dtrain_xgb, val = dval_xgb),
  early_stopping_rounds = 20,
  verbose   = 0
)
cat("Standard XGBoost ESI 1 Binary Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Benchmark Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_xgboost_esi1 <- function(model, dmatrix, actual_factor, set_name) {
  prob_1   <- predict(model, newdata = dmatrix)
  pred_val <- ifelse(prob_1 >= 0.5, "1", "not_1")
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(actual_factor, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  pr_auc  <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  roc_obj <- tryCatch(pROC::roc(act_fac, prob_1, levels = c("not_1", "1")), error = function(e) NULL)
  roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("1", "not_1"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   NATURAL DISTRIBUTION ESI 1 XGBOOST - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 1 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 1 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 1 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 1 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("  ROC-AUC              : %.4f\n", roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, f1 = f1, pr_auc = pr_auc, roc_auc = roc_auc, prob_1 = prob_1, report_df = report_df))
}
res_train <- evaluate_xgboost_esi1(xgb_model, dtrain_xgb, train_df$target_layer1, "Train")
res_val   <- evaluate_xgboost_esi1(xgb_model, dval_xgb,   val_df$target_layer1,   "Validation")
res_test  <- evaluate_xgboost_esi1(xgb_model, dtest_xgb,  test_df$target_layer1,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "xgboost_raw_esi1_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "xgboost_raw_esi1_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_raw_esi1_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_raw_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,  res_val$acc,  res_test$acc),
  Precision = c(res_train$prec, res_val$prec, res_test$prec),
  Recall    = c(res_train$rec,  res_val$rec,  res_test$rec),
  PR_AUC    = c(res_train$pr_auc, res_val$pr_auc, res_test$pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics (Natural Distribution ESI 1 XGBoost)",
       subtitle = "Evaluating ESI 1 Binary XGBoost on Natural Distribution across splits",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_raw_esi1_metrics_barchart.png"), plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_raw_esi1_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Binary XGBoost ESI 1 Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc), file = model_path)
cat(sprintf("Binary ESI 1 XGBoost model saved to: %s\n", model_path))